# Setup
## Dependencies

In [ ]:
%config InlineBackend.figure_format = 'retina'
import sqlite3
from pathlib import Path
from datetime import date
import matplotlib.pyplot as plt
import math

## File Paths

In [ ]:
db_path = Path("../databases/Master_Database.db").resolve()
conn = sqlite3.connect(db_path)

In [ ]:
band_colors = {
    "B": "tab:blue",
    "V": "tab:green",
    "R": "tab:red",
    "r": "tab:orange",
    "I": "tab:purple",
    "C": "tab:black",
}
band_order = ["B", "V", "R", "r", "I", "C"]

## Dictionaries

In [ ]:
discovered_date = {
    "aepk": ("2026-10-06"),
    "abxl": ("2026-09-16"),
    "acna": ("2026-09-20"),
    "aaiv": ("2026-09-01"),
    "aeiv": ("2026-10-05"),
    "aesj": ("2026-10-07"),
    "afsg": ("2026-10-18"),
    "acpp": ("2026-09-21"),
}

# Functions

In [ ]:
def get_supernova(sn_name, conn):
    sql = """
        SELECT obs_date, filter, mag, mag_err
        FROM observations
        WHERE object LIKE ?
            AND mag IS NOT NULL
        ORDER BY obs_date
    """


    sql = """
        SELECT obs_date, filter, mag_f, mag_err_f
        FROM observations
        WHERE object LIKE ?
            AND mag_f IS NOT NULL
        ORDER BY obs_date
    """

    #short_sn_name = sn_name[-4:].lower()
    #output = conn.execute(sql, (short_sn_name,))

    full_name = "%"+sn_name      
    output = conn.execute(sql, (full_name,)).fetchall()

    return (output)

In [ ]:
def get_bands(sn_name, data_list):
    bands_list = {}

    discovery_date = date.fromisoformat(discovered_date[sn_name])
    

    for obs_date, band, mag, mag_err in data_list:
        date_dif = (date.fromisoformat(obs_date)-discovery_date).days

        if mag_err is None:
            mag_err = math.nan

        if band not in bands_list:
            bands_list[band] = {"days": [], "mags": [], "mag_errs": []}

        bands_list[band]["days"].append(date_dif)
        bands_list[band]["mags"].append(mag)
        bands_list[band]["mag_errs"].append(mag_err)

    return bands_list

In [ ]:
data_list = get_supernova("aaiv", conn)

#print(get_bands("aaiv", data_list))


In [ ]:
def plot_figure(sn_name, bands, output_folder):
    
    fig, ax = plt.subplots(figsize=(8, 5))

    for band in band_order:
        if band not in bands:
            continue

        points = bands[band]
        point_color = band_colors[band]

        #ax.plot(points["days"], points["mags"], "o-", color=point_color, label=band)
        ax.errorbar(points["days"], points["mags"], points["mag_errs"], fmt='o-', mfc = 'white', 
                    markersize=5, elinewidth=1, capsize = 3, color=point_color, label = band)

    ax.invert_yaxis()
    ax.set_xlabel("Days Since Discovery")
    ax.set_ylabel("Apparent Magnitude")
    ax.set_ylim(18, 10)
    
    ax.set_title(f"SN 2026{sn_name.upper()} light curve")
    ax.set_title(f"SN 2026{sn_name.upper()} light curve (flat-fielded)")

    
    ax.legend(title="Band")
    

    output_folder.mkdir(parents=True, exist_ok=True)
    
    output_file = output_folder / f"{sn_name}_light_curve.png"
    output_file = output_folder / f"{sn_name}_light_curve_flat.png"

    
    fig.savefig(output_file, dpi=200, bbox_inches="tight")

    plt.show()

    return #output_file




In [ ]:
sn_list = ("aaiv", "acna", "aeiv",
           "aepk", "abxl",  "aesj","acpp", "afsg", 
          )

for sn in sn_list:
    data_list = get_supernova(sn, conn)
    bands = get_bands(sn, data_list)

    plot_figure(sn, bands, Path("../output light curves"))

In [ ]:
for sn_name in discovered_date:
    data_list = get_supernova(sn_name, conn)

    nights = []
    for obs_date, band, mag, mag_err in data_list:
        if obs_date not in nights:
            nights.append(obs_date)

    print(sn_name, ":", len(nights), "nights,", len(data_list), "magnitudes")